In [28]:
import sys
import os


import pandas as pd

sys.path.append(os.path.join(os.path.abspath(""), "../src/python"))

from config import (
    LANGUAGES,
    RESULT_DIR,
)

In [ ]:
def get_language_df(language: str) -> pd.DataFrame:
    base_dir = RESULT_DIR / "lambda_delete_matching" / language
    project_result_paths = list(base_dir.glob("*/**/*.csv"))

    language_df = pd.DataFrame()

    for project_result_path in project_result_paths:
        df = pd.read_csv(project_result_path)

        name_with_owner = (
            f"{project_result_path.parent.name}/{project_result_path.stem}"
        )

        df["name_with_owner"] = name_with_owner

        language_df = pd.concat([language_df, df])

    return language_df

In [30]:
all_df = pd.DataFrame(
    columns=[
        "AR-Projects",
        "Abandoned Replacements",
    ]
)

for language in LANGUAGES:
    df = get_language_df(language)

    project_num = df["name_with_owner"].unique().shape[0]

    exchanged_num = df.shape[0]

    all_df.loc[language] = [
        project_num,
        exchanged_num,
    ]

all_df

,AR-Projects,Abandoned Replacements
java,37,172
javascript,41,183
ruby,5,24
php,5,18
csharp,61,325
cpp,25,164


In [31]:
rq1_df = pd.read_csv(RESULT_DIR / "rq1" / "rq1.csv", index_col=0)

all_df["frac_AR"] = all_df["AR-Projects"] / rq1_df["project_num"]
all_df["frac_Abandoned"] = all_df["Abandoned Replacements"] / rq1_df["replace_count"]

all_df.sort_values("Abandoned Replacements", inplace=True, ascending=False)
all_df.sort_values("AR-Projects", inplace=True, ascending=False)

all_df = all_df[["AR-Projects", "frac_AR", "Abandoned Replacements", "frac_Abandoned"]]

all_df

,AR-Projects,frac_AR,Abandoned Replacements,frac_Abandoned
csharp,61,0.238281,325,0.055131
javascript,41,0.256250,183,0.016731
java,37,0.330357,172,0.042116
cpp,25,0.312500,164,0.083249
ruby,5,0.142857,24,0.137143
php,5,0.277778,18,0.024793


In [32]:
res_rq3_dir = RESULT_DIR / "rq3"
res_rq3_dir.mkdir(parents=True, exist_ok=True)

all_df.to_latex(res_rq3_dir / "rq3.tex")